##### In order to run the bellow cells, download Amazon datasets for electronics from https://amazon-reviews-2023.github.io/main.html and place them in the /data folder.

### Import Dependencies

In [ ]:
import json
import gzip
import pandas as pd

### Working with downloaded data

In [ ]:
with gzip.open("../../data/meta_Electronics.jsonl.gz", "rt") as f:
    first_line = json.loads(f.readline())

In [ ]:
first_line

### Filter out only recent items (appeared in stock in 2022 or 2023)

In [ ]:
def filter_data_by_date(data: dict) -> bool:
    filter = False
    if int(data["details"]["Date First Available"][-4:]) < 2022:
        filter = True

    return filter

In [ ]:
filter_data_by_date(first_line)

In [ ]:
with gzip.open("../../data/meta_Electronics.jsonl.gz", 'rt') as fp:
    with open("../../data/meta_Electronics_2022_2023.jsonl", 'a', encoding='utf-8') as fp_out:
        with open("../../data/meta_Electronics_2022_2023_no_date.jsonl", 'a', encoding='utf-8') as fp_out_no_date:
            i = 0
            for line in fp:
                data = json.loads(line.strip())
                try:
                    filter = filter_data_by_date(data)
                    if not filter:
                        json.dump(data, fp_out)
                        fp_out.write('\n')
                        fp_out.flush()
                except:
                    json.dump(data, fp_out_no_date)
                    fp_out_no_date.write('\n')
                    fp_out_no_date.flush()
                i += 1
                if i % 10000 == 0:
                    print(f"Processed {i} lines")

### Split the items into two categories: "has main category", "does not have main category"

In [ ]:
def filter_category(data: dict) -> dict:
    filter = False
    if data['main_category'] == None:
        filter = True

    return filter

In [ ]:
with open("../../data/meta_Electronics_2022_2023.jsonl", 'r') as fp:
    with open("../../data/meta_Electronics_2022_2023_with_category.jsonl", 'a', encoding='utf-8') as fp_out:
        with open("../../data/meta_Electronics_2022_2023_no_category.jsonl", 'a', encoding='utf-8') as fp_out_no_category:
            i = 0
            for line in fp:
                data = json.loads(line.strip())
                if not filter_category(data):
                    json.dump(data, fp_out)
                    fp_out.write('\n')
                    fp_out.flush()
                else:
                    json.dump(data, fp_out_no_category)
                    fp_out_no_category.write('\n')
                    fp_out_no_category.flush()
                i += 1
                if i % 10000 == 0:
                    print(f"Processed {i} lines")

### Explore distribution by categories

In [ ]:
df = pd.read_json("../../data/meta_Electronics_2022_2023_with_category.jsonl", lines=True)

In [ ]:
df.head()

In [ ]:
len(df)

In [ ]:
df["main_category"].value_counts().plot(kind="bar")

### Filter out items that have at least 100 ratings

In [ ]:
df_ratings_100 = df[df["rating_number"] >= 100]

In [ ]:
len(df)

In [ ]:
len(df_ratings_100)

In [ ]:
df_ratings_100["main_category"].value_counts().plot(kind="bar")

Explore Distribution of ratings

In [ ]:
df_ratings_100["average_rating"].plot(kind="hist", bins=50, range=(0, 5))

### Sample 1000 items

In [ ]:
df_sample_1000 = df_ratings_100.sample(n=1000, random_state=42)

In [ ]:
len(df_sample_1000)

In [ ]:
df_sample_1000["average_rating"].plot(kind="hist", bins=50, range=(0, 5))

In [ ]:
df_sample_1000["price"].plot(kind="hist", bins=100, range=(0, 500))

In [ ]:
df_sample_1000["main_category"].value_counts().plot(kind="bar")

In [ ]:
df_sample_1000.to_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", orient="records", lines=True)

### Test the file

In [ ]:
df_test = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [ ]:
df_test.head()

In [ ]:
len(df_test)